# 01 — Continuous-NHANES correlation block (15 pairs)

Estimates the 15 risk-risk correlations that can be computed in continuous NHANES (1999–2018):

- 10 pairs among `{BMI, LDL-C, SBP, FPG, smoking}`
- 5 pairs of `kidney_dysfunction` (proxied by eGFR via CKD-EPI 2021) against each of the five.

All correlations are weighted Spearman with 95 % CIs from a paired-PSU jackknife. The headline column is the trial-band slice (age 65–80, both sexes pooled) — that's what the simulation will see at simulant initialization. Two stability checks follow:

- **Age stratification** (40–64 / 65–80 / 80+) to see if the correlation is roughly age-invariant or whether the trial-band value is a localized estimate.
- **Time-trend** by 4-year cycle bin (2007–2010 / 2011–2014 / 2015–2018) to see whether the most recent cycles drift away from earlier ones (which would argue for anchoring on the late slice).

In [1]:
import os, warnings
import numpy as np
import pandas as pd
from pathlib import Path
warnings.filterwarnings('ignore')

DATA = Path(os.path.abspath(os.path.join('..', 'data')))
RAW = DATA / 'raw' / 'nhanes'
DERIVED = DATA / 'derived'
OUT = Path('outputs'); OUT.mkdir(parents=True, exist_ok=True)

CYCLE_FILES = {
    '2007_2008': ('SMQ_E.xpt', 'BIOPRO_E.xpt'),
    '2009_2010': ('SMQ_F.xpt', 'BIOPRO_F.xpt'),
    '2011_2012': ('SMQ_G.xpt', 'BIOPRO_G.xpt'),
    '2013_2014': ('SMQ_H.xpt', 'BIOPRO_H.xpt'),
    '2015_2016': ('SMQ_I.xpt', 'BIOPRO_I.xpt'),
    '2017_2018': ('SMQ_J.xpt', 'BIOPRO_J.xpt'),
}

## 1. Pool cycles 2007–2018, derive smoking categories + eGFR

The base parquets at `data/derived/{cycle}.parquet` (built by the `nhanes_vivarium_risk_demo` project) already carry `BMXBMI`, `SBP_MEAN`, `LBDLDL`, `LBXGLU`, demographics, and the masked variance design. We add:

- **Smoking categories.** `SMQ020 == 2` → never; `SMQ020 == 1 & SMQ040 in {1,2}` → current; `SMQ020 == 1 & SMQ040 == 3` → former. Encoded as ordinal 1=current / 2=former / 3=never (matching the project's polytomous PPF: low ordinal = worst stage, so high BMI ↔ likely current smoker → low rank → negative propensity-level correlation).
- **eGFR (CKD-EPI 2021 race-free).** Encoded so **low eGFR = worse kidney function**, matching the project's polytomous PPF (low propensity → cat1 = G5 worst).
- **Combined MEC weight.** Per NCHS analytic guidelines for pooling 2-year cycles, divide `WTMEC2YR` by the number of cycles pooled.

In [2]:
def smoking_cat(row):
    if row['SMQ020'] == 2.0:
        return 3  # never
    if row['SMQ020'] == 1.0:
        if row['SMQ040'] in (1.0, 2.0):
            return 1  # current
        if row['SMQ040'] == 3.0:
            return 2  # former
    return np.nan

def ckd_epi_2021(scr, age, female):
    if pd.isna(scr) or pd.isna(age) or pd.isna(female):
        return np.nan
    kappa = 0.7 if female else 0.9
    alpha = -0.241 if female else -0.302
    sex_factor = 1.012 if female else 1.0
    ratio = scr / kappa
    return 142 * (min(ratio, 1) ** alpha) * (max(ratio, 1) ** -1.200) * (0.9938 ** age) * sex_factor

frames = []
for cyc, (smq_f, biopro_f) in CYCLE_FILES.items():
    df = pd.read_parquet(DERIVED / f'{cyc}.parquet')
    smq = pd.read_sas(RAW / cyc / smq_f)[['SEQN', 'SMQ020', 'SMQ040']]
    bp = pd.read_sas(RAW / cyc / biopro_f)[['SEQN', 'LBXSCR']]
    df = df.merge(smq, on='SEQN', how='left').merge(bp, on='SEQN', how='left')
    df['smoking_cat'] = df.apply(smoking_cat, axis=1)
    df['eGFR'] = df.apply(
        lambda r: ckd_epi_2021(r['LBXSCR'], r['AGE'], r['FEMALE'] == 1.0),
        axis=1,
    )
    frames.append(df)

all_cyc = pd.concat(frames, ignore_index=True)
n_cycles = len(CYCLE_FILES)
all_cyc['weight'] = all_cyc['WTMEC2YR'] / n_cycles
print(f'Pooled {n_cycles} cycles: total respondents = {len(all_cyc):,}')
print(f'  with all 6 risks non-null + weight > 0:')
RISKS = ['BMXBMI', 'LBDLDL', 'SBP_MEAN', 'LBXGLU', 'smoking_cat', 'eGFR']
ok = (all_cyc[RISKS].notna().all(axis=1) & all_cyc['weight'].fillna(0).gt(0))
print(f'    all adults:   n = {ok.sum():,}')
print(f'    age 65–80:    n = {(ok & all_cyc["AGE"].between(65, 80)).sum():,}')

Pooled 6 cycles: total respondents = 36,461
  with all 6 risks non-null + weight > 0:
    all adults:   n = 14,298
    age 65–80:    n = 3,428


## 2. Weighted Spearman + paired-PSU jackknife

Spearman is just Pearson on ranks, so the weighted version uses weighted ranks (each respondent contributes their MEC weight to the cumulative rank). The paired-PSU jackknife replicates the survey-design variance estimator: for each (`SDMVSTRA`, `SDMVPSU`) pair, drop one PSU, double the other, recompute the statistic, accumulate squared deviations from the full-sample value.

In [3]:
def weighted_rank(x, w):
    """Weighted ranks: rank_i = (cumulative weight up through and including
    obs i) − w_i / 2. Ties broken arbitrarily — Spearman is robust to that
    when ties are sparse, and these continuous risks don't have many
    exact ties."""
    o = np.argsort(x, kind='stable')
    cw = np.cumsum(w[o])
    r = np.empty_like(x, dtype=float)
    r[o] = cw - w[o] / 2.0
    return r

def w_spearman(x, y, w):
    rx = weighted_rank(x, w)
    ry = weighted_rank(y, w)
    mx = np.average(rx, weights=w)
    my = np.average(ry, weights=w)
    cov = np.average((rx - mx) * (ry - my), weights=w)
    sx = np.sqrt(np.average((rx - mx) ** 2, weights=w))
    sy = np.sqrt(np.average((ry - my) ** 2, weights=w))
    if sx == 0 or sy == 0:
        return np.nan
    return float(cov / (sx * sy))

def paired_jackknife_spearman(df, x_col, y_col,
                              wt_col='weight',
                              psu_col='SDMVPSU',
                              str_col='SDMVSTRA'):
    sub = df[[x_col, y_col, wt_col, psu_col, str_col]].dropna()
    sub = sub[sub[wt_col] > 0]
    if len(sub) < 30:
        return np.nan, np.nan, len(sub)
    x = sub[x_col].values.astype(float)
    y = sub[y_col].values.astype(float)
    w = sub[wt_col].values.astype(float)
    s = sub[str_col].astype(int).values
    p = sub[psu_col].astype(int).values

    theta = w_spearman(x, y, w)
    var_sum = 0.0
    for stratum in np.unique(s):
        in_str = s == stratum
        psus_in = np.unique(p[in_str])
        if len(psus_in) < 2:
            continue
        for psu in psus_in:
            wr = w.copy()
            mask_in = in_str & (p == psu)
            mask_other = in_str & (p != psu)
            wr[mask_in] = 0.0
            wr[mask_other] = wr[mask_other] * 2.0
            if wr.sum() <= 0:
                continue
            theta_r = w_spearman(x, y, wr)
            if not np.isnan(theta_r):
                var_sum += (theta_r - theta) ** 2
    return theta, np.sqrt(var_sum), len(sub)

## 3. Sign convention — preflight

VPH's polytomous PPF maps **low propensity → cat1 (worst)**. So at the propensity level, *higher exposure* on a continuous risk (e.g. BMI) maps to *higher propensity* on the corresponding GBD-style coding only if we keep the convention `bigger value = worse`.

For the value-axis of each risk, we use:

- **BMI, LDL-C, SBP, FPG**: bigger value → worse (high-cholesterol-bad direction).
- **smoking_cat (1=current, 2=former, 3=never)**: *smaller* ordinal = worst stage. So we **flip** smoking_cat to `4 - smoking_cat` here so the rank correlation reads in the same direction as the others.
- **eGFR**: lower eGFR = worse kidney function (G5 < G4 < ...). For the propensity-level correlation, the project codes `kidney_dysfunction` cat1 (worst) = low propensity, and high eGFR = good (cat5 = TMREL). So eGFR aligns with bigger-is-better, and we **flip** eGFR to `-eGFR` so all six risks have the convention bigger = worse.

Everything below uses the **flipped** versions for the rank correlations. The final matrix output is in the project's propensity-level convention (bigger value = worse on every risk axis), so positive ρ means "the two risks track each other in the worst direction."

In [4]:
all_cyc['smoking_signed']  = 4 - all_cyc['smoking_cat']      # 3=current/worst, 1=never/best
all_cyc['eGFR_signed']     = -all_cyc['eGFR']               # large = worse kidney function

RISK_COLS = {
    'BMI':                  'BMXBMI',
    'LDL_C':                'LBDLDL',
    'SBP':                  'SBP_MEAN',
    'FPG':                  'LBXGLU',
    'smoking':              'smoking_signed',
    'kidney_dysfunction':   'eGFR_signed',
}
RISK_NAMES = list(RISK_COLS.keys())
PAIRS = [(a, b) for i, a in enumerate(RISK_NAMES) for b in RISK_NAMES[i+1:]]
print(f'{len(PAIRS)} unique pairs:')
for a, b in PAIRS: print(f'  {a:<22} ↔ {b}')

15 unique pairs:
  BMI                    ↔ LDL_C
  BMI                    ↔ SBP
  BMI                    ↔ FPG
  BMI                    ↔ smoking
  BMI                    ↔ kidney_dysfunction
  LDL_C                  ↔ SBP
  LDL_C                  ↔ FPG
  LDL_C                  ↔ smoking
  LDL_C                  ↔ kidney_dysfunction
  SBP                    ↔ FPG
  SBP                    ↔ smoking
  SBP                    ↔ kidney_dysfunction
  FPG                    ↔ smoking
  FPG                    ↔ kidney_dysfunction
  smoking                ↔ kidney_dysfunction


## 4. Headline estimates — trial band 65–80, 95 % CIs from paired-PSU jackknife

In [5]:
trial = all_cyc[all_cyc['AGE'].between(65, 80)].copy()
rows = []
for a, b in PAIRS:
    rho, se, n = paired_jackknife_spearman(trial, RISK_COLS[a], RISK_COLS[b])
    rows.append({
        'pair': f'{a} ↔ {b}',
        'risk_a': a, 'risk_b': b,
        'n':   n,
        'rho': rho,
        'se':  se,
        'lo':  rho - 1.96 * se,
        'hi':  rho + 1.96 * se,
    })
headline = pd.DataFrame(rows).round(3)
print('Trial-band 65–80 weighted Spearman, 95 % CI from paired-PSU jackknife:')
print(headline[['pair', 'n', 'rho', 'se', 'lo', 'hi']].to_string(index=False))

Trial-band 65–80 weighted Spearman, 95 % CI from paired-PSU jackknife:
                        pair    n    rho    se     lo     hi
                 BMI ↔ LDL_C 3572 -0.080 0.031 -0.141 -0.019
                   BMI ↔ SBP 7545 -0.038 0.024 -0.085  0.009
                   BMI ↔ FPG 3696  0.290 0.028  0.234  0.345
               BMI ↔ smoking 7811  0.020 0.022 -0.023  0.063
    BMI ↔ kidney_dysfunction 7306  0.046 0.022  0.003  0.089
                 LDL_C ↔ SBP 3519  0.056 0.033 -0.008  0.121
                 LDL_C ↔ FPG 3643 -0.199 0.041 -0.278 -0.119
             LDL_C ↔ smoking 3645 -0.123 0.030 -0.182 -0.064
  LDL_C ↔ kidney_dysfunction 3630 -0.085 0.039 -0.161 -0.009
                   SBP ↔ FPG 3643  0.105 0.029  0.048  0.162
               SBP ↔ smoking 7711 -0.043 0.022 -0.086 -0.000
    SBP ↔ kidney_dysfunction 7215  0.047 0.025 -0.003  0.097
               FPG ↔ smoking 3773  0.066 0.028  0.012  0.121
    FPG ↔ kidney_dysfunction 3711  0.037 0.034 -0.029  0.103
smoking ↔ kidn

## 5. Age stratification — point estimates only

Three age slices (40–64 / 65–80 / 80+). If the three estimates agree within ~0.10, the trial-band value is fine to use uniformly; if they diverge, the project should age-stratify or accept the trial-band as the localized estimate.

In [6]:
AGE_BANDS = {
    '40-64': (40, 65),
    '65-80': (65, 81),
    '80+':   (80, 200),
}
rows = []
for a, b in PAIRS:
    row = {'pair': f'{a} ↔ {b}'}
    for label, (lo, hi) in AGE_BANDS.items():
        sub = all_cyc[all_cyc['AGE'].between(lo, hi - 1)]
        rho, _, n = paired_jackknife_spearman(sub, RISK_COLS[a], RISK_COLS[b])
        row[label] = rho
        row[f'n_{label}'] = n
    rows.append(row)
age_table = pd.DataFrame(rows).round(3)
print('Spearman ρ by age band — divergent if range > 0.10:')
age_table['range'] = (age_table[['40-64', '65-80', '80+']].max(axis=1)
                       - age_table[['40-64', '65-80', '80+']].min(axis=1))
cols = ['pair', 'n_40-64', '40-64', 'n_65-80', '65-80', 'n_80+', '80+', 'range']
print(age_table[cols].round(3).to_string(index=False))
diverging = age_table[age_table['range'] > 0.10]
print(f'\n{len(diverging)} of {len(age_table)} pairs diverge by > 0.10 across age bands.')

Spearman ρ by age band — divergent if range > 0.10:
                        pair  n_40-64  40-64  n_65-80  65-80  n_80+    80+  range
                 BMI ↔ LDL_C     6278  0.009     3572 -0.080    923 -0.071  0.089
                   BMI ↔ SBP    13548  0.192     7545 -0.038   1967 -0.133  0.325
                   BMI ↔ FPG     6542  0.333     3696  0.290    959  0.240  0.093
               BMI ↔ smoking    14052 -0.029     7811  0.020   2062  0.083  0.112
    BMI ↔ kidney_dysfunction    13259  0.038     7306  0.046   1918  0.114  0.076
                 LDL_C ↔ SBP     6124  0.055     3519  0.056    906  0.073  0.018
                 LDL_C ↔ FPG     6343 -0.014     3643 -0.199    948 -0.141  0.185
             LDL_C ↔ smoking     6347  0.005     3645 -0.123    948 -0.145  0.150
  LDL_C ↔ kidney_dysfunction     6332  0.004     3630 -0.085    946 -0.066  0.089
                   SBP ↔ FPG     6380  0.222     3643  0.105    944 -0.007  0.229
               SBP ↔ smoking    13674  0.081  

## 6. Time-trend stratification — point estimates only

Three 4-year cycle bins (2007–2010 / 2011–2014 / 2015–2018), trial band 65–80. The trial period is ~2025–2030, so if the most recent slice (2015–2018) drifts away from earlier ones, anchor on it.

In [7]:
TIME_BINS = {
    '2007-2010': ['2007-2008', '2009-2010'],
    '2011-2014': ['2011-2012', '2013-2014'],
    '2015-2018': ['2015-2016', '2017-2018'],
}
trial_only = trial.copy()
rows = []
for a, b in PAIRS:
    row = {'pair': f'{a} ↔ {b}'}
    for label, cycles in TIME_BINS.items():
        sub = trial_only[trial_only['CYCLE'].isin(cycles)]
        rho, _, n = paired_jackknife_spearman(sub, RISK_COLS[a], RISK_COLS[b])
        row[label] = rho
        row[f'n_{label}'] = n
    rows.append(row)
time_table = pd.DataFrame(rows)
time_table['range'] = (time_table[list(TIME_BINS.keys())].max(axis=1)
                        - time_table[list(TIME_BINS.keys())].min(axis=1))
cols = ['pair'] + sum([['n_'+l, l] for l in TIME_BINS.keys()], []) + ['range']
print('Spearman ρ by 4-year cycle bin (trial band 65–80):')
print(time_table[cols].round(3).to_string(index=False))
drifting = time_table[time_table['range'] > 0.10]
print(f'\n{len(drifting)} of {len(time_table)} pairs drift by > 0.10 across cycle bins.')

Spearman ρ by 4-year cycle bin (trial band 65–80):
                        pair  n_2007-2010  2007-2010  n_2011-2014  2011-2014  n_2015-2018  2015-2018  range
                 BMI ↔ LDL_C         1309     -0.060         1110     -0.092         1153     -0.073  0.032
                   BMI ↔ SBP         2733     -0.050         2281     -0.039         2531     -0.036  0.015
                   BMI ↔ FPG         1332      0.266         1136      0.287         1228      0.296  0.030
               BMI ↔ smoking         2838      0.018         2364     -0.015         2609      0.015  0.033
    BMI ↔ kidney_dysfunction         2654      0.042         2198      0.064         2454      0.042  0.023
                 LDL_C ↔ SBP         1289      0.105         1092      0.059         1138      0.022  0.083
                 LDL_C ↔ FPG         1339     -0.232         1129     -0.095         1175     -0.257  0.162
             LDL_C ↔ smoking         1341     -0.068         1130     -0.131         

## 7. Save for assembly

Save the headline + stratification tables to parquet for the assembly notebook to combine with the Lp(a) and LSM blocks. Also save the prepared multi-cycle DataFrame so notebook 02/03 can reuse it (NHANES III and P_LUX add Lp(a) and LSM but use the same 5 continuous risks).

In [8]:
headline.to_parquet(OUT / 'block_headline.parquet')
age_table.to_parquet(OUT / 'block_age_strat.parquet')
time_table.to_parquet(OUT / 'block_time_strat.parquet')
print(f'wrote {OUT}/block_*.parquet')

wrote outputs/block_*.parquet
